# Pre-entrega: Análisis Visual de Distribuciones Categóricas y Outliers
**Curso:** Data Science II (Módulo 15)  
**Dataset:** Properati - Propiedades en CABA (`properati.csv`)

En este notebook vamos a explorar los datos de propiedades de Properati para:
1. Ver cómo se distribuyen las variables categóricas usando gráficos de barras.
2. Calcular la asimetría (*skewness*) y la curtosis (*kurtosis*) de las variables numéricas.
3. Visualizar y detectar valores atípicos (*outliers*) usando Boxplots, Violin Plots y el método del Rango Intercuartílico (IQR).

## 1. Importación de librerías y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración básica de estilo
sns.set_theme(style='whitegrid')

In [2]:
# Cargamos el archivo CSV
df = pd.read_csv('properati.csv')

# Eliminamos la primera columna de índice si aparece como 'Unnamed: 0'
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Vemos las primeras filas
df.head()

  property_type place_name  price_aprox_usd  rooms  contrafrente  cochera  \
0            PH  Mataderos          62000.0    2.0             0        0   
1     apartment  Mataderos          72000.0    2.0             0        0   
2            PH    Liniers          95000.0    3.0             0        0   
3     apartment   Belgrano         138000.0    1.0             0        0   
4     apartment   Belgrano         195000.0    2.0             0        0   

   amenities  esp_exterior  mas_1banio  estrenar  surface     price_m2  
0          0             1           1         0     55.0  1127.272727  
1          0             0           1         0     55.0  1309.090909  
2          0             1           0         0     57.5  1652.173913  
3          1             0           1         1     45.0  3066.666667  
4          1             0           1         1     65.0  3000.000000  

In [3]:
# Revisamos información general y si hay valores nulos
df.info()
print('\nCantidad de valores nulos por columna:')
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29067 entries, 0 to 29066
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    29067 non-null  object 
 1   place_name       29067 non-null  object 
 2   price_aprox_usd  29067 non-null  float64
 3   rooms            29067 non-null  float64
 4   contrafrente     29067 non-null  int64  
 5   cochera          29067 non-null  int64  
 6   amenities        29067 non-null  int64  
 7   esp_exterior     29067 non-null  int64  
 8   mas_1banio       29067 non-null  int64  
 9   estrenar         29067 non-null  int64  
 10  surface          29067 non-null  float64
 11  price_m2         29067 non-null  float64
dtypes: float64(4), int64(6), object(2)
memory usage: 2.7+ MB

Cantidad de valores nulos por columna:
property_type      0
place_name         0
price_aprox_usd    0
rooms              0
contrafrente       0
cochera            0
amenities          

## 2. Análisis de Variables Categóricas (Gráficos de Barras)
Analizamos las dos variables categóricas principales del dataset:
- `property_type` (Tipo de propiedad)
- `place_name` (Barrio)

In [4]:
# Conteo y porcentaje de cada tipo de propiedad
print('Frecuencia por tipo de propiedad:')
print(df['property_type'].value_counts())

print('\nPorcentaje por tipo de propiedad (%):')
print((df['property_type'].value_counts(normalize=True) * 100).round(2))

Frecuencia por tipo de propiedad:
property_type
apartment    24839
PH            1679
house         1422
store         1127
Name: count, dtype: int64

Porcentaje por tipo de propiedad (%):
property_type
apartment    85.45
PH            5.78
house         4.89
store         3.88
Name: proportion, dtype: float64


In [5]:
# Gráfico de barras 1: Cantidad de propiedades según su tipo
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='property_type', palette='Blues_r')
plt.title('Cantidad de Propiedades por Tipo')
plt.xlabel('Tipo de Propiedad')
plt.ylabel('Cantidad')
plt.show()

In [6]:
# Gráfico de barras 2: Top 10 barrios con más propiedades publicadas
top_10_barrios = df['place_name'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_10_barrios.values, y=top_10_barrios.index, palette='viridis')
plt.title('Top 10 Barrios con Más Propiedades')
plt.xlabel('Cantidad de Propiedades')
plt.ylabel('Barrio')
plt.show()

**Observación:** La gran mayoría de las propiedades en CABA son departamentos (`apartment`, con el 85,45%), mientras que PH, casas y locales representan menos del 15% en conjunto. Los barrios con más publicaciones son Palermo, Belgrano y Caballito.

## 3. Estadísticos Resumen, Asimetría (Skewness) y Curtosis (Kurtosis)
Ahora analizamos las variables numéricas principales: `price_aprox_usd` (precio en USD), `surface` (superficie en m²) y `price_m2` (precio por m²).

In [7]:
# Estadísticas descriptivas básicas
df[['price_aprox_usd', 'rooms', 'surface', 'price_m2']].describe().round(2)

       price_aprox_usd       rooms     surface    price_m2
count         29067.00    29067.00    29067.00    29067.00
mean         228084.00        2.58       92.59     2531.02
std          241890.92        1.37       94.66      902.11
min           22400.00        1.00       18.00     1000.00
25%           99715.00        1.00       42.00     1976.47
50%          148000.00        2.00       60.00     2409.50
75%          255000.00        3.00      104.00     2900.00
max         3000000.00       10.00     2750.00     9215.69

In [8]:
# Cálculo de Asimetría (Skewness) y Curtosis (Kurtosis)
print('--- Asimetría (Skewness) ---')
print('price_aprox_usd:', round(df['price_aprox_usd'].skew(), 2))
print('surface:        ', round(df['surface'].skew(), 2))
print('price_m2:       ', round(df['price_m2'].skew(), 2))

print('\n--- Curtosis (Kurtosis) ---')
print('price_aprox_usd:', round(df['price_aprox_usd'].kurt(), 2))
print('surface:        ', round(df['surface'].kurt(), 2))
print('price_m2:       ', round(df['price_m2'].kurt(), 2))

--- Asimetría (Skewness) ---
price_aprox_usd: 3.92
surface:         5.86
price_m2:        1.76

--- Curtosis (Kurtosis) ---
price_aprox_usd: 22.20
surface:         85.86
price_m2:        6.44


### Interpretación de los valores obtenidos:
- **Asimetría positiva (cola hacia la derecha):** Las tres variables tienen valores de *skewness* positivos mayores a 0 (`3.92` en precio, `5.86` en superficie y `1.76` en precio por m²). Esto indica que la mayoría de los inmuebles tienen precios y tamaños moderados, pero existen unas pocas propiedades muy caras o muy grandes que estiran la distribución hacia la derecha (por eso la media de precio es USD 228.084 mientras que la mediana es USD 148.000).
- **Curtosis alta (distribución leptocúrtica):** Los valores de curtosis son muy altos (especialmente en `price_aprox_usd` con `22.20` y `surface` con `85.86`). Esto significa que los datos están muy concentrados cerca del centro pero tienen colas pesadas con muchos valores extremos (*outliers*).

## 4. Detección de Outliers: Boxplots, Violin Plots y Método IQR
Primero visualizamos las distribuciones y sus valores extremos usando **Boxplots** (diagramas de caja) y **Violin Plots** (diagramas de violín).

In [9]:
# Boxplot del precio en USD según el tipo de propiedad
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='property_type', y='price_aprox_usd', palette='Set2')
plt.title('Boxplot: Distribución del Precio (USD) por Tipo de Propiedad')
plt.xlabel('Tipo de Propiedad')
plt.ylabel('Precio en USD')
plt.show()

In [10]:
# Violin Plot del precio por m2 según el tipo de propiedad
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='property_type', y='price_m2', palette='Set3')
plt.title('Violin Plot: Distribución del Precio por m² según Tipo de Propiedad')
plt.xlabel('Tipo de Propiedad')
plt.ylabel('Precio por m² (USD)')
plt.show()

### Cálculo estadístico de Outliers con el Rango Intercuartílico (IQR)
Aplicamos la fórmula vista en clase sobre la variable `price_aprox_usd`:
- $IQR = Q_3 - Q_1$
- $\text{Límite Inferior} = Q_1 - 1.5 \times IQR$
- $\text{Límite Superior} = Q_3 + 1.5 \times IQR$

In [11]:
# 1. Calculamos Q1, Q3 e IQR para el precio (price_aprox_usd)
Q1 = df['price_aprox_usd'].quantile(0.25)
Q3 = df['price_aprox_usd'].quantile(0.75)
IQR = Q3 - Q1

# 2. Definimos los límites inferior y superior
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print('Q1 (25%): USD', Q1)
print('Q3 (75%): USD', Q3)
print('IQR:      USD', IQR)
print('Límite Inferior: USD', limite_inferior)
print('Límite Superior: USD', limite_superior)

# 3. Filtramos y listamos los registros que son outliers
outliers_precio = df[(df['price_aprox_usd'] < limite_inferior) | (df['price_aprox_usd'] > limite_superior)]

print('\nCantidad de outliers detectados en precio:', len(outliers_precio))
print('Porcentaje sobre el total del dataset:', round(len(outliers_precio) / len(df) * 100, 2), '%')

# Mostramos los primeros 10 registros considerados outliers
outliers_precio[['property_type', 'place_name', 'price_aprox_usd', 'rooms', 'surface', 'price_m2']].head(10)

Q1 (25%): USD 99715.0
Q3 (75%): USD 255000.0
IQR:      USD 155285.0
Límite Inferior: USD -133212.5
Límite Superior: USD 487927.5

Cantidad de outliers detectados en precio: 2575
Porcentaje sobre el total del dataset: 8.86 %


   property_type   place_name  price_aprox_usd  rooms  surface     price_m2
13         house        Boedo         550000.0    6.0    504.0  1091.269841
27         house       Flores         650000.0    5.0    450.0  1444.444444
43     apartment      Palermo         540000.0    4.0    165.0  3272.727273
44     apartment      Palermo         890000.0    4.0    210.0  4238.095238
52     apartment     Recoleta         560000.0    4.0    205.0  2731.707317
57     apartment     Belgrano         490000.0    4.0    140.0  3500.000000
60     apartment      Palermo         580000.0    4.0    160.0  3625.000000
61     apartment      Palermo         620000.0    4.0    175.0  3542.857143
66     apartment     Recoleta         750000.0    5.0    240.0  3125.000000
75     apartment     Belgrano         520000.0    4.0    150.0  3466.666667

In [12]:
# Repetimos el cálculo de IQR para la variable superficie (surface)
Q1_sup = df['surface'].quantile(0.25)
Q3_sup = df['surface'].quantile(0.75)
IQR_sup = Q3_sup - Q1_sup

lim_inf_sup = Q1_sup - 1.5 * IQR_sup
lim_sup_sup = Q3_sup + 1.5 * IQR_sup

outliers_superficie = df[(df['surface'] < lim_inf_sup) | (df['surface'] > lim_sup_sup)]

print(f'Q1 superficie: {Q1_sup} m2 | Q3 superficie: {Q3_sup} m2 | IQR: {IQR_sup} m2')
print(f'Límite Superior de superficie: {lim_sup_sup} m2')
print(f'Cantidad de outliers en superficie: {len(outliers_superficie)} ({round(len(outliers_superficie)/len(df)*100, 2)} %)')

Q1 superficie: 42.0 m2 | Q3 superficie: 104.0 m2 | IQR: 62.0 m2
Límite Superior de superficie: 197.0 m2
Cantidad de outliers en superficie: 2624 (9.03 %)


## 5. Conclusiones: ¿Qué decisión tomar con los outliers encontrados?

Luego de analizar los gráficos y los registros filtrados con el método IQR, concluimos lo siguiente:

1. **No eliminar todos los outliers directamente:** Vemos que los 2.575 outliers de precio (propiedades por encima de USD 487.927) representan casi el 9% de la base y corresponden a propiedades reales: casas grandes, locales comerciales o departamentos de lujo en barrios caros como Palermo, Recoleta, Belgrano y Puerto Madero. Si los borramos todos, perderíamos información real de ese segmento del mercado.
2. **Aplicar una transformación logarítmica:** Como las variables `price_aprox_usd` y `surface` tienen una fuerte asimetría positiva (cola larga a la derecha), una buena decisión antes de entrenar un modelo de Machine Learning es aplicarles el logaritmo (`np.log10` o `np.log1p`) para achicar el efecto de los valores extremos y lograr una distribución más simétrica.
3. **Separar el análisis según el tipo de propiedad (`property_type`):** En el Boxplot vimos que las casas y los locales tienen precios y superficies mucho más altos que los departamentos. Por eso conviene analizar o filtrar los outliers por separado dentro de cada categoría.